## Combined Run Analysis — Deep Dive

This notebook analyses the **combined** dataset run in detail. The combined run differs from the per-dataset runs in two important ways:

1. **CNN encoder**: trained on whuGAIT subjects (IDs 21–118, 98-class identification) and then **frozen**. The same CNN is reused without fine-tuning across all datasets.
2. **Test subjects**: there are no held-out whuGAIT subjects — all 98 whuGAIT subjects are used for training. The test set consists entirely of **ucihar** and **wisdm** subjects that the CNN has never encountered.

This creates a natural cross-dataset generalisation challenge: the frozen CNN was optimised for whuGAIT motion patterns, and the LSTM must learn to compare subjects from different sensor domains.

### Test dataset breakdown

| Group | IDs | n subjects |
|---|---|---|
| ucihar held-out | 1001–1030 | 30 |
| wisdm held-out | 2001–2051 | 51 |

Sections below break down authenticator performance, score distributions, and training curves for each group. MIA and evasion sections are shown as placeholders until those pipeline stages run for the combined dataset.

In [ ]:
import sys, re, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve, auc as sk_auc

import torch
sys.path.insert(0, '../..')
from src.models.gait_cnn  import GaitCNN
from src.models.auth_model import AuthModel
from src.data.auth_dataset import normalize_auth

LOG_DIR      = Path('../../logs/combined')
ARTIFACT_DIR = Path('../../artifacts/combined')
CKPT_DIR     = Path('../../checkpoints/combined')
EXEC_DIR     = Path('../../executed/combined')
TEX_DIR      = Path('../../latex/generated')
OUT_DIR      = Path('../../results/analysis')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cpu')
BATCH  = 512

split = json.load(open(ARTIFACT_DIR / 'subject_split.json'))
held_out_ucihar = set(split['held_out_ucihar'])
held_out_wisdm  = set(split['held_out_wisdm'])
print(f'ucihar held-out: {len(held_out_ucihar)} subjects  IDs {min(held_out_ucihar)}–{max(held_out_ucihar)}')
print(f'wisdm  held-out: {len(held_out_wisdm)} subjects  IDs {min(held_out_wisdm)}–{max(held_out_wisdm)}')

In [ ]:
# Load raw test pairs and normalise
ap = np.load(ARTIFACT_DIR / 'auth_pairs.npz')
X1_te_raw = ap['X1_te']
X2_te_raw = ap['X2_te']
y_te      = ap['y_te'].astype(int)
subj1_te  = ap['subj1_te']
subj2_te  = ap['subj2_te']

ns = np.load(ARTIFACT_DIR / 'auth_norm_stats.npz')
X1_te, X2_te, _ = normalize_auth(X1_te_raw, X2_te_raw, stats=(ns['mean'], ns['std']))

print(f'Test pairs: {len(y_te):,}  same={int((y_te==0).sum()):,}  diff={int((y_te==1).sum()):,}')
print(f'subj1 range: {subj1_te.min()}–{subj1_te.max()}')

# Per-pair dataset mask: a pair belongs to a source if BOTH subjects are from that source
ucihar_mask = np.array([s1 in held_out_ucihar and s2 in held_out_ucihar
                        for s1, s2 in zip(subj1_te, subj2_te)])
wisdm_mask  = np.array([s1 in held_out_wisdm  and s2 in held_out_wisdm
                        for s1, s2 in zip(subj1_te, subj2_te)])
cross_mask  = ~ucihar_mask & ~wisdm_mask
print(f'ucihar-only pairs: {ucihar_mask.sum():,}  wisdm-only: {wisdm_mask.sum():,}  cross: {cross_mask.sum():,}')

In [ ]:
# Load checkpoint and run inference
meta = json.load(open(CKPT_DIR / 'cnn_encoder_meta.json'))
cnn  = GaitCNN(n_classes=meta['n_classes'])
cnn.load_state_dict(torch.load(CKPT_DIR / 'cnn_encoder.pt', map_location='cpu'))

model = AuthModel(cnn_encoder=cnn, dropout=0.3).to(DEVICE)
model.load_state_dict(torch.load(CKPT_DIR / 'auth_model.pt', map_location='cpu'))
model.eval()

X1_t = torch.from_numpy(X1_te)
X2_t = torch.from_numpy(X2_te)
scores_list = []
preds_list  = []
with torch.no_grad():
    for s in range(0, len(X1_t), BATCH):
        x1b = X1_t[s:s+BATCH].to(DEVICE)
        x2b = X2_t[s:s+BATCH].to(DEVICE)
        logits = model(x1b, x2b)
        scores_list.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        preds_list.append(logits.argmax(1).cpu().numpy())

scores = np.concatenate(scores_list)   # P(different person)
preds  = np.concatenate(preds_list)

overall_acc = (preds == y_te).mean()
overall_auc = roc_auc_score(y_te, scores)
print(f'Overall: acc={overall_acc*100:.2f}%  AUC={overall_auc:.4f}')

## Authenticator performance — ucihar vs wisdm test sets

The combined model must generalise from whuGAIT sensor data to two different sensor domains. These metrics show whether the cross-domain transfer is uniform or biased toward one of the two held-out datasets.

In [ ]:
def subset_metrics(mask, label='overall'):
    if mask.sum() == 0:
        return {'label': label, 'n_pairs': 0, 'acc': np.nan, 'auc': np.nan, 'sep': np.nan}
    s, p, y = scores[mask], preds[mask], y_te[mask]
    acc = (p == y).mean()
    auc = roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan
    sep = s[y == 1].mean() - s[y == 0].mean()
    return {'label': label, 'n_pairs': int(mask.sum()), 'acc': acc, 'auc': auc, 'sep': sep}

rows = [
    subset_metrics(np.ones(len(scores), dtype=bool), 'overall'),
    subset_metrics(ucihar_mask, 'ucihar'),
    subset_metrics(wisdm_mask,  'wisdm'),
    subset_metrics(cross_mask,  'cross-dataset'),
]
df_perf = pd.DataFrame(rows).set_index('label')
print(df_perf.round(4).to_string())

# Bar chart: accuracy and AUC per group
groups  = ['overall', 'ucihar', 'wisdm']
colors  = ['#9b59b6', '#e74c3c', '#2ecc71']
w, x    = 0.35, np.arange(len(groups))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (metric, ylabel, title) in zip(axes, [
    ('acc',  'Accuracy', 'Test Accuracy — combined model'),
    ('auc',  'AUC',      'Test AUC — combined model'),
]):
    vals = [df_perf.loc[g, metric] for g in groups]
    bars = ax.bar(x, vals, color=colors, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([f'{g}\n(n={df_perf.loc[g,"n_pairs"]:,})' for g in groups])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, 1.1 if metric == 'auc' else 110)
    ax.axhline(0.5 if metric == 'auc' else 50, ls='--', color='grey', alpha=0.5, label='random')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            lbl = f'{val:.3f}' if metric == 'auc' else f'{val*100:.1f}%'
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    lbl, ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'combined_per_dataset_perf.png', dpi=150)
plt.show()

## Score distributions — ucihar vs wisdm

Histograms of P(different person) for same-person and different-person pairs, split by test source dataset. A well-calibrated model should show clear separation (same-person scores near 0, different-person near 1). Overlap indicates confusion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (mask, label, color) in zip(axes, [
    (ucihar_mask, 'ucihar', '#e74c3c'),
    (wisdm_mask,  'wisdm',  '#2ecc71'),
]):
    if mask.sum() == 0:
        ax.text(0.5, 0.5, f'No {label} pairs', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{label} — no data')
        continue
    s_sub = scores[mask]
    y_sub = y_te[mask]
    same  = s_sub[y_sub == 0]
    diff  = s_sub[y_sub == 1]
    ax.hist(same, bins=40, alpha=0.65, color='#2ecc71', density=True,
            label=f'Same person (n={len(same):,})')
    ax.hist(diff, bins=40, alpha=0.65, color='#e74c3c', density=True,
            label=f'Diff person (n={len(diff):,})')
    ax.axvline(0.5, ls='--', color='black', alpha=0.7, label='threshold 0.5')
    sep = diff.mean() - same.mean()
    acc_sub = ((s_sub >= 0.5).astype(int) == y_sub).mean()
    auc_sub = roc_auc_score(y_sub, s_sub)
    ax.set_xlabel('P(different person)')
    ax.set_ylabel('Density')
    ax.set_title(f'{label} — acc={acc_sub*100:.1f}%  AUC={auc_sub:.3f}  sep={sep:.3f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Score distributions — combined model (ucihar vs wisdm test sets)', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'combined_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Training curves — epoch-by-epoch

Parsed from the executed NB03 output. The combined model trains on 98 whuGAIT subjects and is validated on the mixed ucihar + wisdm test set. The large train–test gap reflects the domain shift from whuGAIT (training domain) to the cross-dataset test subjects.

In [ ]:
EPOCH_PATTERN = (
    r'Epoch\s+(?P<epoch>\d+)/\d+'
    r'.*?train_acc=(?P<train_acc>[\d.]+)%'
    r'.*?test_acc=(?P<test_acc>[\d.]+)%'
    r'.*?train_loss=(?P<train_loss>[\d.]+)'
    r'.*?test_loss=(?P<test_loss>[\d.]+)'
)

def parse_epoch_curves(nb_path, pattern):
    import re as _re
    if not nb_path.exists():
        return []
    nb = json.load(open(nb_path))
    rows = []
    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        for out in cell.get('outputs', []):
            txt = ''.join(out.get('text', ''))
            for line in txt.split('\n'):
                m = _re.search(pattern, line)
                if m:
                    rows.append({k: float(v) for k, v in m.groupdict().items()})
    return rows

curves = parse_epoch_curves(EXEC_DIR / '03_train_authenticator.ipynb', EPOCH_PATTERN)

if curves:
    best_ep = int(max(curves, key=lambda r: r['test_acc'])['epoch'])
    best_test_acc = max(r['test_acc'] for r in curves)
    epochs     = [r['epoch']      for r in curves]
    train_acc  = [r['train_acc']  for r in curves]
    test_acc   = [r['test_acc']   for r in curves]
    train_loss = [r['train_loss'] for r in curves]
    test_loss  = [r['test_loss']  for r in curves]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.plot(epochs, train_acc,  color='#9b59b6', lw=2,        label='Train acc')
    ax.plot(epochs, test_acc,   color='#9b59b6', lw=2, ls='--', label='Test acc')
    ax.axvline(best_ep, ls=':', color='black', alpha=0.5, label=f'Best epoch ({best_ep})')
    ax.fill_between(epochs, train_acc, test_acc, alpha=0.15, color='orange', label='Gap')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Train vs Test Accuracy — combined')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, train_loss, color='#9b59b6', lw=2,        label='Train loss')
    ax.plot(epochs, test_loss,  color='#9b59b6', lw=2, ls='--', label='Test loss')
    ax.axvline(best_ep, ls=':', color='black', alpha=0.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
    ax.set_title('Train vs Test Loss — combined')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'combined_training_curves.png', dpi=150)
    plt.show()
    print(f'Best epoch: {best_ep}  test_acc: {best_test_acc:.2f}%')
else:
    print('Training curves not found — run the combined pipeline first.')

## MIA results

_Not yet executed for the combined run. Run NB04–NB05c for dataset=combined and re-execute this notebook._

In [ ]:
mia_report    = ARTIFACT_DIR / '05b_per_subject_report.json'
target_deltas = ARTIFACT_DIR / '05a_target_deltas.npz'

if mia_report.exists() and target_deltas.exists():
    from scipy.stats import norm as sp_norm
    from sklearn.metrics import roc_auc_score, roc_curve

    report = json.load(open(mia_report))
    subjects   = report['per_subject']
    delta_vals = np.array([s['Simple-D']['score'] for s in subjects])
    labels_mia = np.array([1 if s['true_label'] == 'member' else 0 for s in subjects])

    simple_d_auc = roc_auc_score(labels_mia, delta_vals)
    print(f'Simple-D AUC: {simple_d_auc:.4f}')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(delta_vals[labels_mia == 1], bins=20, alpha=0.7, label='members',     color='#3498db', density=True)
    ax.hist(delta_vals[labels_mia == 0], bins=20, alpha=0.7, label='non-members', color='#e74c3c', density=True)
    ax.set_xlabel('δ score'); ax.set_ylabel('Density')
    ax.set_title(f'MIA delta distribution — combined  (AUC={simple_d_auc:.3f})')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'combined_mia_deltas.png', dpi=150)
    plt.show()
else:
    print('MIA results not available yet for combined run.')
    print(f'  Missing: {mia_report if not mia_report.exists() else target_deltas}')

## Evasion attack results

_Not yet executed for the combined run. Run NB06a–NB06b for dataset=combined and re-execute this notebook._

In [ ]:
evasion_path = ARTIFACT_DIR / '06b_combined_attack_results.npz'

if evasion_path.exists():
    d  = np.load(evasion_path, allow_pickle=True)
    succ = d['pair_succeeded'].astype(bool)
    psr  = float(np.nanmean(d['pair_psr']))
    asr  = float(d['combo_asr'].mean())
    full = int((d['combo_asr'] == 1.0).sum())
    eps_t = float(d['eps_target'])
    resist = int((~succ).sum())

    print(f'Evasion results — combined')
    print(f'  budget_ratio (PSR):  {psr:.4f}  ({psr*100:.1f}% of ε_target needed)')
    print(f'  combo ASR:           {asr:.3f}  ({full}/{len(d["combo_asr"])} full combos)')
    print(f'  resistant pairs:     {resist}/{len(succ)}')
    print(f'  ε_target:            {eps_t:.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    eps_min = d['pair_eps_min'][succ]
    axes[0].hist(eps_min / eps_t, bins=30, color='#9b59b6', alpha=0.8, density=True)
    axes[0].axvline(1.0, ls='--', color='black', alpha=0.7, label='ε_target')
    axes[0].axvline(psr, ls='-',  color='red',   alpha=0.7, label=f'mean={psr:.3f}')
    axes[0].set_xlabel('ε_min / ε_target'); axes[0].set_ylabel('Density')
    axes[0].set_title('Budget ratio distribution'); axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    asr_vals = d['combo_asr']
    axes[1].hist(asr_vals, bins=20, color='#9b59b6', alpha=0.8)
    axes[1].set_xlabel('Combo ASR'); axes[1].set_ylabel('Count')
    axes[1].set_title(f'Combo ASR distribution  mean={asr:.3f}')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'combined_evasion.png', dpi=150)
    plt.show()
else:
    print('Evasion results not available yet for combined run.')
    print(f'  Missing: {evasion_path}')

## Summary

In [ ]:
mia_auc      = None
evasion_asr  = None
budget_ratio = None

if (ARTIFACT_DIR / '05b_per_subject_report.json').exists():
    report   = json.load(open(ARTIFACT_DIR / '05b_per_subject_report.json'))
    subjects = report['per_subject']
    _d  = np.array([s['Simple-D']['score'] for s in subjects])
    _lb = np.array([1 if s['true_label'] == 'member' else 0 for s in subjects])
    mia_auc = roc_auc_score(_lb, _d)

if (ARTIFACT_DIR / '06b_combined_attack_results.npz').exists():
    _ev = np.load(ARTIFACT_DIR / '06b_combined_attack_results.npz', allow_pickle=True)
    evasion_asr  = float(_ev['combo_asr'].mean())
    budget_ratio = float(np.nanmean(_ev['pair_psr']))

rows = []
for grp, mask in [('overall', np.ones(len(scores), dtype=bool)),
                  ('ucihar',  ucihar_mask),
                  ('wisdm',   wisdm_mask)]:
    m = subset_metrics(mask, grp)
    rows.append({
        'group':        grp,
        'n_pairs':      m['n_pairs'],
        'auth_acc':     f'{m["acc"]*100:.2f}%' if not np.isnan(m['acc']) else '—',
        'auth_auc':     f'{m["auc"]:.4f}'      if not np.isnan(m['auc']) else '—',
        'score_sep':    f'{m["sep"]:.3f}'       if not np.isnan(m['sep']) else '—',
        'mia_auc':      f'{mia_auc:.4f}'        if mia_auc      else '(pending)',
        'evasion_asr':  f'{evasion_asr:.3f}'    if evasion_asr  else '(pending)',
        'budget_ratio': f'{budget_ratio:.3f}'   if budget_ratio else '(pending)',
    })

df_summary = pd.DataFrame(rows).set_index('group')
print('=== Combined Run Summary ===')
print(df_summary.to_string())